# A5b · MiniMax-M3 MSA (블록 top-k sparse attention) 검증

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_llm_dit/blob/main/notebooks/verify/A5b_minimax_msa.ipynb)

원본
- 정답 함수: github `MiniMax-AI/MSA` @ `80434d7` 의 순수 PyTorch reference
  - `sparse_attention_ref` (`python/fmha_sm100/cute/test_sparse_atten.py`): 고른 블록만 보는 GQA attention. 커널 테스트의 golden.
  - `ref_tile_max_score` (`tests/regression/test_correctness.py`): 128 토큰 블록별 최대 q·k 점수 (index 가지의 블록 점수).
  - 두 파일은 커널 패키지를 import 하므로 import 하지 않고, 함수 소스만 잘라 실행합니다.
- 층 연결: sglang `models/minimax_m3.py` (투영 이름, gemma RMSNorm, partial RoPE, index 가지)와
  `minimax_sparse/prefill/flash_with_topk_idx.py` 의 top-k 규칙(local 블록 강제, 오름차순, -1 채움)을 옮겨 적은 `RefMSAAttention`.
- HF `MiniMaxAI/MiniMax-M3` 는 config 만 있고 modeling 코드가 없어서, **모델 전체가 아니라 attention 층 단위로 비교**합니다.

내 구현: `miniarch/models/minimax_msa.py` (`MSAAttention`, 작은 LM `MiniMaxMSA`).

부품
- Main 가지: GQA + head 별 gemma RMSNorm + partial RoPE
- Index 가지: 그룹당 index query 1개 + 공유 index key 1개 → 블록 max 점수 → 그룹별 top-k (local 블록 강제)
- Main 가지는 고른 블록 안의 causal 토큰만 attention

축소 config 선택 이유
- `ref_tile_max_score` 가 블록 크기 128을 고정해 두어서 block 128, 시퀀스 512 → 블록 4개.
- topk 2 (= local 1 + 고르는 블록 1): 블록 0 질의는 볼 블록이 topk보다 적어 -1 채움 경로, 블록 1은 전부 선택,
  블록 2·3은 실제 선택이 일어납니다. KV head 2개 → 그룹 2개가 서로 다른 블록을 고르는지도 비교합니다.
- hook 비교: 블록 점수(`index_score`), 고른 블록(`block_select`), 최종 출력.
- 내 구현은 attention 층이라 자동 causal 검사 대신, 자체 점검에서 LM 전체의 causal 누수를 검사합니다.

순서: 0. 환경 준비 → 1. 원본 구조 (HF 헤더) → 2. 원본 모델 동작 확인 → 3. 내 구현 == 원본 (층별 출력 비교) → 4. 학습 동작 확인 → 5. 단계 전용 자체 점검 → 6. 결과 저장

**런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 위에서부터 실행하세요. 다른 모델 노트북과 독립적으로 돌아갑니다.

## 0. 환경 준비

In [ ]:
import os
REPO_DIR = "/content/tmp_llm_dit"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/HisameOgasahara/tmp_llm_dit.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
if not os.environ.get("MINIARCH_SKIP_PIP"):   # 전체 실행 노트북이 한 번만 설치하고 넘길 때 건너뜀
    !pip -q install -U -r requirements-colab.txt
    # Colab 기본 torchao 가 최신 diffusers 와 버전이 안 맞아 import 가 깨짐 → 쓰지 않으므로 제거
    !pip -q uninstall -y torchao

In [ ]:
import sys, torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (런타임 유형을 GPU로 바꾸세요)")
!git log -1 --format='repo commit: %h %s'

In [ ]:
from miniarch import verify as V
from miniarch.verify import stages
S = stages.get("A5b")
dev = "cuda" if torch.cuda.is_available() else "cpu"
try:   # gated 모델 헤더/원격 코드용. Colab 왼쪽 🔑(보안 비밀)에 HF_TOKEN 등록.
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

CHECKS = []   # (이름, 통과, 오차) → 마지막에 저장
def report(name, ok, err=float("nan")):
    CHECKS.append({"check": name, "ok": bool(ok), "max_abs": float(err)})
    print(f"{'✅' if ok else '❌'} {name:<44} max_abs={err:.2e}")

## 원본 구조 (HF 헤더)

실제 체크포인트의 safetensors 헤더만 읽어 가중치 이름, shape, dtype을 봅니다 (다운로드 없음).
**구현 검증이 아니라 원본 구조 확인용**입니다. routed expert는 개수가 많아 숨깁니다.

In [ ]:
def peek(repo, pattern, subfolder="", limit=60):
    print(f"\n===== {repo}  /{pattern}/")
    try:
        cfg = V.fetch_config(repo, subfolder)
        cfg = cfg.get("text_config", cfg)
        keys = ["model_type", "hidden_size", "num_hidden_layers", "num_attention_heads", "num_key_value_heads", "head_dim"]
        print({k: cfg[k] for k in keys if k in cfg})
        h = V.read_model_header(repo, subfolder=subfolder, pattern=pattern)
        V.show({k: v for k, v in h.items() if ".experts." not in k}, limit)
    except Exception as e:
        print(f"⚠️ {type(e).__name__}: {e}")

peek('MiniMaxAI/MiniMax-M3', 'language_model\\.model\\.layers\\.(0|3)\\.(self_attn|input_layernorm)', subfolder='')

## 원본 모델 동작 확인

축소 config로 원본을 만들고 한 번 forward 합니다. 여기서 실패하면 라이브러리 버전 문제입니다 (내 구현과 무관).

In [ ]:
V.set_deterministic(0)
ref, inputs = S.ref()
ref = ref.to(dev).eval()
with torch.no_grad():
    out = ref(**{k: v.to(dev) if torch.is_tensor(v) else v for k, v in inputs.items()})
t = out[0] if isinstance(out, tuple) else getattr(out, "last_hidden_state", None)
t = t if t is not None else out.sample
print(f"{type(ref).__name__}  params={sum(p.numel() for p in ref.parameters()):,}  out={tuple(t.shape)}")

## 내 구현 == 원본

원본 가중치에 잡음을 넣고(초기값 0/1 때문에 복사 누락이 가려지지 않도록) 내 구현에 복사한 뒤, 층별 출력과 최종 출력을 비교합니다.

In [ ]:
row = V.cases.run_case(S.CASE)
print(row["status"]); print(row["detail"])
report("원본 동치 (" + S.CASE.reference + ")", row["status"].startswith("✅"))

## 학습 동작 확인

forward 가 원본과 같아도 학습이 되는지는 따로 확인해야 합니다.
- **기울기 == 원본**: 같은 가중치·같은 손실로 원본과 내 구현을 backward 해서 파라미터 기울기를 비교합니다 (파라미터별 상대 오차 ≤ 1e-3).
  원본이 추론 전용 코드(no_grad)면 건너뜁니다.
- **학습 궤적 == 원본**: 같은 가중치·데이터·optimizer(SGD momentum 0.9, 30 step)로 원본과 내 구현을 나란히 학습해
  step 별 손실과 최종 가중치를 비교합니다 (상대차 ≤ 1e-3). Adam 은 아주 작은 잡음 차이를 키워서 궤적 비교에 쓰지 않습니다.
- **배치 하나 외우기**: 고정 배치 하나로 300 step 학습해 손실이 처음의 20% 이하로 내려가는지 봅니다 (LM은 다음 토큰 CE, 그 외는 고정 목표 MSE).
  NaN/inf 가 없어야 하고, 기울기를 한 번도 받지 못한 파라미터를 보여줍니다 (MoE 에서 안 뽑힌 expert 는 정상).

In [ ]:
import importlib, traceback
from miniarch.verify import training as TR
mod = importlib.import_module(S.CASE.mine_module)
V.set_deterministic(0)
ref, inputs = S.ref()
V.perturb_params(ref)
pair = mod.verify_pair(ref)
try:
    g = TR.grad_equivalence(ref, pair, inputs, dev)
    if g.note.startswith("원본 출력에 기울기가 없음"):
        print("⏭️", g.note); report("기울기 == 원본 (원본 추론 전용 → 건너뜀)", True)
    else:
        print(f"비교한 파라미터 {g.compared}개, 최대 상대 오차 {g.max_rel:.2e} ({g.worst})", g.note)
        if g.skipped_ref:
            print("원본에서 기울기 없는 파라미터:", g.skipped_ref[:8])
        report("기울기 == 원본", g.ok, g.max_rel)
except Exception:
    traceback.print_exc(limit=4); report("기울기 == 원본", False)

In [ ]:
try:
    t = TR.train_parity(ref, pair, inputs, dev)
    if t.note.startswith("원본 출력에 기울기가 없음"):
        print("⏭️", t.note); report("학습 궤적 == 원본 (원본 추론 전용 → 건너뜀)", True)
    else:
        print(f"{t.steps} step  손실 최대 상대차 {t.max_loss_rel:.2e}  최종 가중치 최대 상대차 {t.max_param_rel:.2e} ({t.worst_param})", t.note)
        print("  원본:", " ".join(f"{v:.4g}" for v in t.loss_ref[::6]))
        print("  내것:", " ".join(f"{v:.4g}" for v in t.loss_mine[::6]))
        report("학습 궤적 == 원본 (SGD 30 step)", t.ok, max(t.max_loss_rel, t.max_param_rel))
except Exception:
    traceback.print_exc(limit=4); report("학습 궤적 == 원본", False)

In [ ]:
try:
    o = TR.overfit_batch(pair["mine"], pair, inputs, dev)
    print(f"{o.kind}: {o.loss_start:.4f} → {o.loss_end:.4f}  (유한={o.finite})")
    print("  손실:", " ".join(f"{v:.3g}" for v in o.curve[::30]))
    if o.no_grad_params:
        print(f"  기울기를 못 받은 파라미터 {len(o.no_grad_params)}개:", o.no_grad_params[:8])
    report("배치 하나 외우기 (" + o.kind + ")", o.ok, o.loss_end / max(o.loss_start, 1e-12))
except Exception:
    traceback.print_exc(limit=4); report("배치 하나 외우기", False)

### 원본과 비교할 수 없는 부분

- **수치 미분 (float64)**: 무작위 방향으로 가중치를 ±ε 움직여 잰 손실 변화가 autograd 기울기와 같은지 봅니다.
  원본이 추론 전용이라 기울기를 비교할 수 없는 모델도 이걸로 backward 를 확인합니다.
- **라우터 bias 업데이트 (aux-loss-free)**: 원본 추론 코드에는 없는 학습 전용 규칙입니다. 많이 뽑힌 expert 의 bias 는 내려가고
  적게 뽑힌 쪽은 올라가는지, bias 가 optimizer 에서 빠져 있는지, bias 가 실제로 선택을 바꾸는지 봅니다. 라우터 bias 가 없는 모델은 건너뜁니다.

In [ ]:
try:
    f = TR.directional_grad_check(pair["mine"], pair, inputs, dev)
    print(f"방향 {len(f.rel_errors)}개 상대 오차 {[f'{e:.1e}' for e in f.rel_errors]}  (ε={f.eps})")
    report("수치 미분 == autograd (float64)", f.ok, max(f.rel_errors))
except Exception:
    traceback.print_exc(limit=4); report("수치 미분 == autograd (float64)", False)
try:
    b = TR.router_bias_check(pair["mine"], pair, inputs, dev)
    print("\n".join(b.details))
    report("라우터 bias 업데이트" if b.n_routers else "라우터 bias 업데이트 (라우터 bias 없음 → 건너뜀)", b.ok)
except Exception:
    traceback.print_exc(limit=4); report("라우터 bias 업데이트", False)

## 단계 전용 자체 점검

### 내 LM 전체 causal 누수

dense 1층 + MSA 1층 LM. 시퀀스 800 (블록 7개)의 뒤쪽 절반(400~)을 바꿔도 앞쪽 출력이 그대로여야 합니다.
앞쪽 질의 블록 2·3은 실제로 블록을 골라야 하고, 블록 3(384~511)은 과거·미래 토큰이 섞여 있어
블록 점수가 질의 위치까지의 키만 써야 통과합니다.

In [ ]:
from miniarch.models import minimax_msa as M
V.set_deterministic(0)
c = M.config_from_hf(S.hf_config())
lm = M.MiniMaxMSA(c).to(dev).eval()
r = V.causal_leak_test(lambda x: lm(x).last_hidden_state, vocab_size=c.vocab_size, seq_len=800, device=dev, atol=1e-4)
report("내 LM " + r.name, r.ok, r.max_abs)

### topk ≥ 블록 수 → dense attention 과 같음 / topk 작으면 달라짐

모든 블록을 고르면 MSA는 보통 causal GQA와 같아야 합니다 (`use_sparse=False` 와 비교).
반대로 topk 2면 결과가 달라져야 sparse 경로가 실제로 쓰인 것입니다.

In [ ]:
import dataclasses
from miniarch.models import minimax_msa as M
V.set_deterministic(0)
c = M.config_from_hf(S.hf_config())
full = M.MiniMaxMSA(dataclasses.replace(c, topk=16)).to(dev).eval()
dense = M.MiniMaxMSA(dataclasses.replace(c, use_sparse=False)).to(dev).eval()
small = M.MiniMaxMSA(c).to(dev).eval()
dense.load_state_dict(full.state_dict()); small.load_state_dict(full.state_dict())
x = torch.randint(0, c.vocab_size, (2, 400), device=dev)
r = V.same_function_test("topk 16 (블록 4개) vs dense", lambda: full(x).logits, lambda: dense(x).logits, atol=1e-5)
report(r.name, r.ok, r.max_abs)
with torch.no_grad():
    d = (small(x).logits - dense(x).logits).abs().max().item()
report("topk 2 는 dense 와 달라야 함", d > 1e-4, d)

### 블록 선택 · attention map hook

init 블록 1개, topk 3, 시퀀스 600 (블록 5개)으로 바꿔 init 강제 경로까지 봅니다.
- 고른 블록 수 = min(topk, 질의 블록 + 1), init 블록(0)과 local 블록(질의가 속한 블록)은 항상 포함
- `attn_probs` 는 고른 블록 밖에서 0, 행 합 1

In [ ]:
import dataclasses
from miniarch.models import minimax_msa as M
V.set_deterministic(0)
c = dataclasses.replace(M.config_from_hf(S.hf_config()), init_blocks=1, topk=3)
lm = M.MiniMaxMSA(c).to(dev).eval()
i = c.layer_types.index("sparse_attention")
pre = f"layers.{i}.self_attn."
x = torch.randint(0, c.vocab_size, (2, 600), device=dev)
with V.capture(lm, [pre + "block_select", pre + "attn_probs", pre + "index_score"]) as st, torch.no_grad():
    lm(x)
sel, p = st[pre + "block_select"], st[pre + "attn_probs"]          # [B, G, T, NB], [B, H, T, T]
T, bs = x.shape[1], c.block_size
qb = torch.arange(T, device=dev) // bs
print("block_select", tuple(sel.shape), " attn_probs", tuple(p.shape), " index_score", tuple(st[pre + "index_score"].shape))
report("고른 블록 수 = min(topk, 질의 블록+1)", bool((sel.sum(-1) == (qb + 1).clamp(max=c.topk)).all()))
report("init 블록 항상 포함", bool(sel[..., 0].all()))
report("local 블록 항상 포함", bool(sel.gather(-1, qb.view(1, 1, T, 1).expand(*sel.shape[:3], 1)).all()))
tok = sel.repeat_interleave(bs, -1)[..., :T].repeat_interleave(c.n_head // c.n_kv_head, 1)
leak = p.masked_fill(tok, 0).abs().max().item()
report("고른 블록 밖 확률 = 0", leak == 0, leak)
row = (p.sum(-1) - 1).abs().max().item()
report("attention 행 합 = 1", row < 1e-5, row)
last = int(qb.max())
for g in range(sel.shape[1]):   # 그룹마다 고르는 블록이 다른지 (마지막 질의 블록 기준, 블록별 선택 비율)
    print(f"그룹 {g}:", [round(v, 2) for v in sel[:, g, qb == last].float().mean((0, 1)).tolist()])

### index 가지 KL 손실 · 기울기 격리

`index_kl=True` 로 학습 모드 forward → `kl_loss.backward()`.
teacher(main 확률)는 detach, index 입력은 stopgrad(x) 이므로 기울기는 index 투영·norm 에만 가야 합니다 (논문 식 11).

In [ ]:
import dataclasses
from miniarch.models import minimax_msa as M
V.set_deterministic(0)
c = dataclasses.replace(M.config_from_hf(S.hf_config()), index_kl=True)
lm = M.MiniMaxMSA(c).to(dev).train()
x = torch.randint(0, c.vocab_size, (2, 400), device=dev)
lm(x)
kl = lm.kl_loss
kl.backward()
leak = [n for n, q in lm.named_parameters() if ".index_" not in n and q.grad is not None and q.grad.abs().max() > 0]
got = [n for n, q in lm.named_parameters() if ".index_" in n and q.grad is not None and q.grad.abs().max() > 0]
print("KL =", kl.item(), " 기울기 받은 index 파라미터:", got)
report("KL ≥ 0 이고 유한", bool(torch.isfinite(kl)) and kl.item() >= -1e-6, kl.item())
report("KL 기울기는 index 가지에만", len(leak) == 0 and len(got) > 0, float(len(leak)))

## 결과 저장

- `results/`에 점검 결과를 저장합니다.
- 노트북 출력까지 남기려면 **파일 → GitHub에 사본 저장** → 이 저장소의 `notebooks/verify/A5b_minimax_msa.ipynb`.

In [ ]:
import datetime, pandas as pd
os.makedirs("results", exist_ok=True)
df = pd.DataFrame(CHECKS)
p = f"results/verify_A5b_{datetime.datetime.now():%Y%m%d-%H%M}.csv"
df.to_csv(p, index=False)
print(df.to_string(index=False)); print("saved:", p)
print("전체:", "✅ 모두 통과" if df.ok.all() else "❌ 실패 있음")